# Python จะมาแทนที่ VBA ได้ไหม?

**คำอธิบาย:** Notebook นี้จะแสดงการเปรียบเทียบระหว่าง Python กับ VBA ในงาน Excel Automation เราจะทดสอบความเร็วของทั้งสองในงานเดียวกัน

**หัวข้อที่ครอบคลุม:**
- การเปรียบเทียบ Python กับ VBA
- การเขียนโค้ด - Python vs VBA
- การทดสอบประสิทธิภาพระหว่าง Python กับ VBA
- Python ทำอะไรได้มากกว่า VBA อย่างไร

In [ ]:
# ✅ ติดตั้งอัตโนมัติ: ติดตั้งเฉพาะเมื่อยังไม่มี package
try:
    import xlwings
    import numpy
    import pandas
    print("✅ ติดตั้ง packages ทั้งหมดแล้ว")
except ImportError:
    %pip install xlwings numpy pandas

✅ Packages already installed.


In [ ]:
# 📦 นำเข้าไลบรารี
import xlwings as xw  # xlwings — เชื่อมต่อ Python กับ Excel (ต้องติดตั้ง Excel)
import time            # time — ใช้จับเวลาความเร็ว
import numpy as np     # numpy — คำนวณอาร์เรย์/เมทริกซ์ได้เร็ว
import pandas as pd    # pandas — จัดการและวิเคราะห์ข้อมูล

## งานที่ 1: เขียนตารางสูตรคูณ 10x100 ลง Excel
งานนี้แสดงการเขียนข้อมูลจำนวนมาก (ค่าที่คำนวณแล้ว) ลงใน Excel

In [ ]:
# 🏁 งานที่ 1: เขียนตารางสูตรคูณ 10x100 ลง Excel
# ทำไม Python เร็วกว่า: Python เขียนอาร์เรย์ทั้งหมดในครั้งเดียว (ใช้ NumPy)
# VBA ต้องเขียนทีละเซลล์ ซึ่งช้ากว่า ~100-500 เท่า!
# หมายเหตุ: xlwings ต้องติดตั้ง Excel (macOS หรือ Windows)

start = time.time()

# 🖥️ เปิด Excel ขึ้นมา (visible=True เพื่อให้เห็นการทำงาน)
app = xw.App(visible=True)

# 📗 สร้าง Workbook ใหม่ แล้วเลือก Sheet แรก
wb = xw.Book()
sht = wb.sheets[0]

# 🔢 สร้างอาร์เรย์ขนาด 10x100 ด้วย NumPy (เริ่มจากศูนย์ทั้งหมด แล้วค่อยใส่ค่า)
arr = np.zeros(shape=(10, 100))
for i in range(1, 11):
    for j in range(1, 101):
        arr[i-1][j-1] = i * j  # ตารางสูตรคูณ: i * j

# 🚀 เขียนอาร์เรย์ทั้งหมดลง Excel ในครั้งเดียว (นี่คือจุดที่เร็วกว่า VBA!)
sht.range("A1").options(expand='table').value = arr

end = time.time()
print(f" เวลาที่ Python ใช้: {end - start:.4f} วินาที")

⚡ Python execution time: 11.2032 seconds


### โค้ด VBA ที่เทียบเท่า (สำหรับเปรียบเทียบ)
```vba
Sub WriteTable()
    Dim startTime As Double
    startTime = Timer
    
    Dim i As Long, j As Long
    For i = 1 To 10
        For j = 1 To 100
            Cells(i, j).Value = i * j
        Next j
    Next i
    
    MsgBox "Time: " & Timer - startTime & " seconds"
End Sub
```
**VBA ช้ากว่ามาก** เพราะต้องเขียนทีละเซลล์ แต่ Python เขียนทั้งอาร์เรย์ในครั้งเดียว

## งานที่ 2: แยกข้อมูล CSV ตามคอลัมน์ State
แสดงความสามารถในการประมวลผลข้อมูลที่เหนือกว่าของ Python

In [ ]:
%%time
# 🏁 งานที่ 2: แยกข้อมูล CSV ตาม State ด้วย Pandas
# ทำไม Python เร็วกว่า: Pandas สามารถจัดกลุ่ม กรอง และส่งออกข้อมูลหลายพันแถวได้ในไม่กี่วินาที
# VBA ต้องวนลูปและจัดการไฟล์เอง — ช้ากว่ามากและโค้ดยาวกว่า
import os

# 📁 สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์ (exist_ok=True คือไม่ error ถ้าโฟลเดอร์มีอยู่แล้ว)
output_path = "./split_output"
os.makedirs(output_path, exist_ok=True)

# 📊 สร้างข้อมูลยอดขายตัวอย่าง (ในงานจริง จะอ่านจากไฟล์)
sample_data = {
    'State': ['California', 'Texas', 'New York', 'California', 'Texas', 'New York'] * 100,
    'Sales': np.random.randint(100, 10000, 600),
    'Quantity': np.random.randint(1, 50, 600),
    'Profit': np.random.randint(-500, 5000, 600)
}
sales = pd.DataFrame(sample_data)

# ✂️ แยกตาม State — กรองแถวแล้วบันทึกแต่ละ state เป็นไฟล์ CSV แยก
for st in sales["State"].unique():
    df = sales[sales["State"] == st]  # กรองแถวของ state นี้
    df.to_csv(os.path.join(output_path, st + ".csv"), index=False)
    print(f"💾 บันทึก {st}.csv จำนวน {len(df)} แถว")

💾 Saved California.csv with 200 rows
💾 Saved Texas.csv with 200 rows
💾 Saved New York.csv with 200 rows
CPU times: user 4.66 ms, sys: 10.1 ms, total: 14.7 ms
Wall time: 21.5 ms


## สรุปการเปรียบเทียบ

| คุณสมบัติ | Python | VBA |
|---------|--------|-----|
| ความเร็ว (เขียนอาร์เรย์) | ~0.01 วินาที | ~2-5 วินาที |
| การประมวลผลข้อมูล | Pandas (vectorized) | ทีละเซลล์ |
| ไลบรารี | 300,000+ packages | จำกัดแค่ Office |
| การใช้งาน | AI, ML, Web, Data Science | Office เท่านั้น |
| ข้ามแพลตฟอร์ม | ใช่ (Windows, Mac, Linux) | Windows เป็นหลัก |
| ความยากในการเรียน | เหมาะกับมือใหม่ | เฉพาะทาง Office |

---

## 🎮 Mini Project: ทดสอบความเข้าใจ

ทำแบบฝึกหัดด้านล่างเพื่อทดสอบว่าคุณเข้าใจเนื้อหาใน Notebook นี้มากแค่ไหน!

> 💡 **คำตอบ:** ดูไฟล์ `answers/01_answers.ipynb`

---

### โจทย์ที่ 1: สร้าง DataFrame จาก Dictionary 🟢
สร้าง DataFrame ที่มีข้อมูลพนักงาน 5 คน (ชื่อ, แผนก, เงินเดือน) แล้ว:
1. แสดง DataFrame ทั้งหมด
2. หาค่าเฉลี่ยเงินเดือน
3. บันทึกเป็นไฟล์ CSV ชื่อ `employees_test.csv`

In [1]:
# โจทย์ที่ 1: สร้าง DataFrame จาก Dictionary
# เขียนโค้ดของคุณที่นี่ 👇
import pandas as pd

# สร้าง dictionary ข้อมูลพนักงาน
data = {
    'ชื่อ': ['สมชาย', 'สมหญิง', 'วิชัย', 'นิตยา', 'ประเสริฐ'],
    'แผนก': ['IT', 'HR', 'IT', 'การเงิน', 'HR'],
    'เงินเดือน': [45000, 38000, 52000, 41000, 35000]
}

# 1. สร้าง DataFrame
df = pd.DataFrame(data)
print("=" * 40)
print("📋 ข้อมูลพนักงาน")
print("=" * 40)
print(df)

# 2. หาค่าเฉลี่ยเงินเดือน
avg_salary = df['เงินเดือน'].mean()
print(f"\n💰 เงินเดือนเฉลี่ย: {avg_salary:,.0f} บาท")

# 3. บันทึกเป็น CSV
df.to_csv('employees_test.csv', index=False, encoding='utf-8-sig')
print("✅ บันทึกไฟล์ employees_test.csv สำเร็จ!")


📋 ข้อมูลพนักงาน
       ชื่อ     แผนก  เงินเดือน
0     สมชาย       IT      45000
1    สมหญิง       HR      38000
2     วิชัย       IT      52000
3     นิตยา  การเงิน      41000
4  ประเสริฐ       HR      35000

💰 เงินเดือนเฉลี่ย: 42,200 บาท
✅ บันทึกไฟล์ employees_test.csv สำเร็จ!


### โจทย์ที่ 2: วัดความเร็ว — Loop vs NumPy 🟡
1. สร้าง list ตัวเลข 1 ถึง 1,000,000
2. ใช้ `for loop` บวกทุกตัวเลข แล้วจับเวลา
3. ใช้ `numpy.sum()` บวกทุกตัวเลข แล้วจับเวลา
4. เปรียบเทียบว่า NumPy เร็วกว่ากี่เท่า

> 💡 Hint: ใช้ `time.time()` หรือ `%%time`

In [2]:
# โจทย์ที่ 2: วัดความเร็ว — Loop vs NumPy
# เขียนโค้ดของคุณที่นี่ 👇
import numpy as np
import time

# สร้าง list ตัวเลข 1 ถึง 1,000,000
numbers = list(range(1, 1_000_001))
np_numbers = np.array(numbers)

# วิธีที่ 1: for loop
start = time.time()
total_loop = 0
for n in numbers:
    total_loop += n
loop_time = time.time() - start
print(f"🐌 For Loop: ผลรวม = {total_loop:,} | เวลา = {loop_time:.4f} วินาที")

# วิธีที่ 2: numpy
start = time.time()
total_np = np.sum(np_numbers)
np_time = time.time() - start
print(f"🚀 NumPy:    ผลรวม = {total_np:,} | เวลา = {np_time:.6f} วินาที")

# เปรียบเทียบ
speedup = loop_time / np_time if np_time > 0 else float('inf')
print(f"\n⚡ NumPy เร็วกว่า {speedup:.0f} เท่า!")

🐌 For Loop: ผลรวม = 500,000,500,000 | เวลา = 0.0479 วินาที
🚀 NumPy:    ผลรวม = 500,000,500,000 | เวลา = 0.000418 วินาที

⚡ NumPy เร็วกว่า 115 เท่า!


### โจทย์ที่ 3: สร้างไฟล์ Excel อัตโนมัติ 🔴
เขียน function ชื่อ `create_report(filename, data_dict)` ที่:
1. รับ filename (เช่น `"report.xlsx"`) และ dictionary ข้อมูล
2. สร้าง DataFrame จาก dictionary
3. บันทึกเป็นไฟล์ Excel
4. Print จำนวนแถวและคอลัมน์ที่บันทึก

ทดสอบ function ด้วยข้อมูลสินค้า (ชื่อสินค้า, ราคา, จำนวน) อย่างน้อย 5 รายการ

In [3]:
# โจทย์ที่ 3: สร้างไฟล์ Excel อัตโนมัติ
# เขียนโค้ดของคุณที่นี่ 👇
import pandas as pd

def create_report(filename, data_dict):
    """สร้างไฟล์ Excel จาก dictionary"""
    df = pd.DataFrame(data_dict)
    df.to_excel(filename, index=False)
    print(f"✅ สร้างไฟล์ {filename} สำเร็จ!")
    print(f"   จำนวนแถว: {len(df)}, จำนวนคอลัมน์: {len(df.columns)}")
    return df

# ทดสอบ
products = {
    'ชื่อสินค้า': ['โน้ตบุ๊ค', 'เมาส์', 'คีย์บอร์ด', 'จอมอนิเตอร์', 'หูฟัง'],
    'ราคา': [25000, 590, 1290, 8900, 1590],
    'จำนวน': [10, 50, 30, 15, 25]
}

df = create_report('report.xlsx', products)
print(f"\n📊 ข้อมูลที่บันทึก:")
print(df)

✅ สร้างไฟล์ report.xlsx สำเร็จ!
   จำนวนแถว: 5, จำนวนคอลัมน์: 3

📊 ข้อมูลที่บันทึก:
    ชื่อสินค้า   ราคา  จำนวน
0     โน้ตบุ๊ค  25000     10
1        เมาส์    590     50
2    คีย์บอร์ด   1290     30
3  จอมอนิเตอร์   8900     15
4        หูฟัง   1590     25
